## Guided project: counting protesters — who counts, and how many?

Every big demonstration in France produces **two numbers**: one from the organisers (usually the CGT or the intersyndicale) and one from the Interior Ministry (via the police). On **17 October 2026**, the cost-of-living mobilisation brought out **800,000 people according to the organisers and 550,000 according to the Interior Ministry**, including 110,000 in Paris by the ministry's count.

There is **no official open dataset** of these counts: they are published day by day in press releases and news reports. In this project you will build that dataset yourself (session 6: scraping), wrangle it (session 7: MultiIndex, `apply`/`map`, time series), and analyse it.

**Possible research questions** (refine them, or bring your own, if you do, discuss your hypotheses with us first):
1. How large is the gap between the two counts, and is it stable over a movement?
2. Is the gap larger on big days than on small ones? (*H: organisers and police diverge more as crowds grow.*)
3. Does Paris weigh the same in both counts? (*H: organisers report a larger Paris share.*)
4. How does protest activity in France compare with other European countries over time? (ACLED data, section 5)

**You can decide to work on this topic for your project**

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)

### 1. A seed dataset: the 2023 pension-reform movement

The 2023 movement against the pension reform had **14 national days of mobilisation**. The table below gives both counts for France and, where reported, for Paris. Every row has a source you can check.

Things to notice before you analyse anything:
* `lower_bound = True` means the figure was reported as *"plus de…"* (more than).
* **6 April 2023 is missing on purpose**: finding it is your first data-collection task (section 2).
* Sources can disagree: for 15 March, one outlet reports 1.7 million for the CGT and another 1.78 million. Note which one you use.

In [2]:
VIKI = "https://fr.vikidia.org/wiki/Mouvement_social_contre_la_r%C3%A9forme_des_retraites_en_France_de_2023"
ENWIKI = "https://en.wikipedia.org/wiki/2023_French_pension_reform_strikes"

rows = [
    # day, date, organisers_fr, authorities_fr, organisers_paris, authorities_paris, lower_bound, source
    (1,  "2023-01-19", 2_000_000, 1_120_000, 400_000,  80_000, False, VIKI),
    (2,  "2023-01-31", 2_800_000, 1_272_000, 500_000,  87_000, False, VIKI),
    (3,  "2023-02-07", 2_000_000,   757_000, 400_000,  57_000, False, VIKI),
    (4,  "2023-02-11", 2_500_000,   963_000, 500_000,  93_000, False, VIKI),
    (5,  "2023-02-16", 1_300_000,   440_000, None,       None, False, ENWIKI),
    (6,  "2023-03-07", 3_500_000, 1_280_000, 700_000,  81_000, False, VIKI),
    (7,  "2023-03-11", 1_000_000,   368_000, None,       None, False,
         "https://www.blick.ch/fr/monde/entre-368000-et-un-million-de-manifestants-la-mobilisation-contre-la-reforme-des-retraites-faiblit-en-france-id18390697.html"),
    (8,  "2023-03-15", 1_700_000,   480_000, 450_000,  37_000, False,
         "https://www.ici.fr/infos/economie-social/reforme-des-retraites-suivez-la-huitieme-journee-de-mobilisation-4045118 ; Paris: " + ENWIKI),
    (9,  "2023-03-23", 3_500_000, 1_089_000, 800_000, 119_000, False, VIKI),
    (10, "2023-03-28", 2_000_000,   740_000, 450_000,  93_000, False, VIKI),
    (11, "2023-04-06", None,           None, None,       None, False, "TO COLLECT"),
    (12, "2023-04-13", 1_500_000,   380_000, None,       None, True,
         "https://www.franceinfo.fr/economie/retraite/reforme-des-retraites/direct-reforme-des-retraites-12e-journee-de-mobilisation-a-la-veille-de-la-decision-du-conseil-constitutionnel_5768261.html"),
    (13, "2023-05-01", 2_300_000,   782_000, None,     112_000, False,
         "https://www.europe1.fr/societe/1er-mai-782000-manifestants-en-france-selon-linterieur-23-millions-selon-la-cgt-4180852 ; Paris: https://www.noovo.info/nouvelle/plus-de-782-000-manifestants-dans-les-rues-en-france-dont-112-000-a-paris.html"),
    (14, "2023-06-06",   900_000,   281_000, None,       None, True,
         "https://www.europe1.fr/societe/mobilisation-du-6-juin-281000-manifestants-en-france-selon-linterieur-plus-de-900000-selon-la-cgt-4187333"),
]
cols = ["day", "date", "organisers_fr", "authorities_fr", "organisers_paris",
        "authorities_paris", "lower_bound", "source"]
pension = pd.DataFrame(rows, columns=cols)
pension["movement"] = "Pension reform 2023"

current = pd.DataFrame([
    (1, "2026-10-17", 800_000, 550_000, None, 110_000, False,
     "https://www.franceinfo.fr/economie/greve/mobilisation-du-17-octobre-800-000-personnes-ont-manifeste-dans-toute-la-france-selon-les-organisateurs-550-000-selon-le-ministre-de-l-interieur_8242081.html ; Paris: https://encause.fr/mobilisation-17-octobre-2026-bilan-chiffres-violences/"),
], columns=cols)
current["movement"] = "Cost of living 2026"

print(pension.drop(columns="source"))

    day        date  organisers_fr  authorities_fr  organisers_paris  authorities_paris  lower_bound  \
0     1  2023-01-19      2000000.0       1120000.0          400000.0            80000.0        False   
1     2  2023-01-31      2800000.0       1272000.0          500000.0            87000.0        False   
2     3  2023-02-07      2000000.0        757000.0          400000.0            57000.0        False   
3     4  2023-02-11      2500000.0        963000.0          500000.0            93000.0        False   
4     5  2023-02-16      1300000.0        440000.0               NaN                NaN        False   
5     6  2023-03-07      3500000.0       1280000.0          700000.0            81000.0        False   
6     7  2023-03-11      1000000.0        368000.0               NaN                NaN        False   
7     8  2023-03-15      1700000.0        480000.0          450000.0            37000.0        False   
8     9  2023-03-23      3500000.0       1089000.0          8000

### 2. Collecting more data: parsing news headlines

Most counts are published in headlines like the ones below (real headlines from franceinfo, ici, Europe 1). To extend the dataset you need a function that turns a headline into two numbers. The formats are messy on purpose: `900 000`, `480.000`, `1,7 million`, `3,5 millions`, `1,08` with no unit at all, and the two counts can appear in either order.

In [4]:
HEADLINES = [
    "Réforme des retraites : 900 000 personnes ont manifesté en France, selon la CGT, 281 000 selon le ministère de l'Intérieur, lors de la 14e journée de mobilisation",
    "Retraites : 3,5 millions de manifestants en France selon la CGT, 1,08 selon le ministère de l'Intérieur",
    "Réforme des retraites : 1,7 million de manifestants selon la CGT, 480.000 d'après le ministère de l'Intérieur",
    "Réforme des retraites : 2,8 millions de manifestants en France selon la CGT, 1,27 million d'après la police",
    "Mobilisation du 6 juin : 281.000 manifestants en France selon l'Intérieur, plus de 900.000 selon la CGT",
    "La mobilisation du 17 octobre a rassemblé entre 800 000 personnes dans toute la France selon les organisateurs et 550 000 d'après le ministre de l'Intérieur",
]

# Given: a pattern for numbers, and a helper that converts one match to an int
NUMBER = re.compile(r"(\d+(?:[ .,]\d+)*)\s*(millions?)?")

def to_number(digits, million):
    """'1,7' + 'million' -> 1_700_000 ; '480.000' -> 480_000 ; '1,08' (no unit) -> 1_080_000."""
    if million or ("," in digits and len(digits.split(",")[-1]) <= 2):
        return round(float(digits.replace(" ", "").replace(".", "").replace(",", ".")) * 1_000_000)
    return int(re.sub(r"[ .,]", "", digits))

print(to_number("1,7", "million"), to_number("480.000", None), to_number("1,08", None))

1700000 480000 1080000


##### Exercise 1

Write `parse_counts(headline)` that returns a tuple `(organisers, authorities)`.

Hint: loop over `NUMBER.finditer(headline)`. For each number, look at the text **between this number and the next one**: if it mentions the CGT, the unions or the organisers, it is the organisers' count; if it mentions the Interior, the ministry or the police, it is the authorities' count. Numbers with no such words around them (like *14e journée* or *6 juin*) are ignored.

Test: `for h in HEADLINES: print(parse_counts(h))`

In [ ]:
# Write your exercise here

Expected output:

```text
(900000, 281000)
(3500000, 1080000)
(1700000, 480000)
(2800000, 1270000)
(900000, 281000)
(800000, 550000)
```

In [ ]:
# Write your solution here

##### Exercise 2

Day 11 (**6 April 2023**) is missing. Find a news report with both counts for that day, check that `parse_counts()` reads it correctly, and fill in the row in `pension` (keep the source URL!). Use `requests` + `BeautifulSoup` from session 6 if you want to collect headlines automatically — and respect each site's `robots.txt`.

In [ ]:
# Write your exercise here

##### Exercise 3

The 2026 cost-of-living movement is still going. Each time there is a new national day, add a row to `current` with both counts (France and Paris when available) and its source. Can you also find figures for the student and education strike of **13 October 2026**?

In [ ]:
# Write your exercise here

### 3. Wrangling: ratios, shares and a MultiIndex

Combine the movements, convert the dates, and compute two indicators:
* `ratio_fr` — organisers ÷ authorities (how many times larger the organisers' count is)
* `paris_share_org` and `paris_share_auth` — Paris as a share of the national count, for each counter

In [ ]:
num_cols = ["organisers_fr", "authorities_fr", "organisers_paris", "authorities_paris"]
protests = pd.concat([df.astype({c: "float" for c in num_cols}) for df in (pension, current)],
                     ignore_index=True)
protests["date"] = pd.to_datetime(protests["date"])

protests["ratio_fr"] = protests["organisers_fr"] / protests["authorities_fr"]
protests["paris_share_org"] = protests["organisers_paris"] / protests["organisers_fr"]
protests["paris_share_auth"] = protests["authorities_paris"] / protests["authorities_fr"]

shares = ["ratio_fr", "paris_share_org", "paris_share_auth"]
print(protests[["movement", "date"]].join(protests[shares].round(2)))

The same data in **long format** with a MultiIndex `(movement, date)` on the rows and `(scope, counter)` on the columns makes comparisons easy:

In [ ]:
wide = protests.set_index(["movement", "date"])[num_cols]
wide.columns = pd.MultiIndex.from_tuples(
    [("France", "organisers"), ("France", "authorities"), ("Paris", "organisers"), ("Paris", "authorities")],
    names=["scope", "counter"])
print(wide.loc["Pension reform 2023"].head())
print(wide.xs("authorities", level="counter", axis=1).tail(3))

##### Exercise 4

Using `wide`:
1. Select only the Paris columns with `.xs("Paris", level="scope", axis=1)`.
2. Use `apply(..., axis=1)` to label each day `"organisers ≥ 3× authorities"` or `"below 3×"`, then count the labels with `value_counts()`.
3. Which day of the 2023 movement had the largest gap in absolute numbers? And in ratio?

In [ ]:
# Write your exercise here

### 4. First exploration

Two starting plots. They are descriptive: your project should go further, with a clear question and a careful reading of the data's limits (two counts, two methods, some lower bounds).

In [ ]:
p = protests[protests["movement"] == "Pension reform 2023"].set_index("date")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(p[["organisers_fr", "authorities_fr"]] / 1e6).plot(ax=axes[0], marker="o")
axes[0].set_title("Pension reform 2023: protesters in France")
axes[0].set_ylabel("Millions")
axes[0].set_xlabel("")

axes[1].scatter(p["authorities_fr"] / 1e6, p["ratio_fr"])
axes[1].set_title("Is the gap larger on bigger days?")
axes[1].set_xlabel("Authorities' count (millions)")
axes[1].set_ylabel("Organisers ÷ authorities")
plt.tight_layout()

In [ ]:
print("Correlation between size (authorities) and ratio:",
      round(p["authorities_fr"].corr(p["ratio_fr"]), 2))
print("Mean Paris share, organisers: ", round(p["paris_share_org"].mean(), 3))
print("Mean Paris share, authorities:", round(p["paris_share_auth"].mean(), 3))

##### Exercise 5

1. What do the two plots suggest about questions 1 and 2 from the introduction? With 13 data points, how confident can you be?
2. Compare the two Paris shares. Write two sentences on what could explain the difference (counting methods? where each counter stands? which marches are included?).
3. What would you need to collect to test your explanation?

In [ ]:
# Write your answer here (as comments) or in a markdown cell

### 5. The world: ACLED protest events

For other countries, the main source is **ACLED** (Armed Conflict Location & Event Data), which codes every demonstration as an event with a date, a location and actors, worldwide.

**Getting the data.** Create a free myACLED account at [acleddata.com](https://acleddata.com), open the *Data Export Tool*, choose **event-level data**, event type **Protests** (and Riots if relevant), the countries you want, and download the CSV. Save it next to this notebook as `acled_protests.csv`.

**Read the limits before you analyse:**
* At the Research access level, event-level data comes with a **12-month lag**, so the most recent months (including October 2026) may not be there yet.
* ACLED has **no exact crowd-size variable**. The `tags` column contains a bucket: `crowd size=very small` (<20), `small` (20–99), `medium` (100–999), `large` (1,000–9,999) or `massive` (10,000+). Events with no information default to `small`.
* For a quick qualitative overview of major antigovernment protests worldwide, see the [Carnegie Global Protest Tracker](https://carnegieendowment.org/features/global-protest-tracker).

In [ ]:
ACLED_FILE = Path("acled_protests.csv")

if ACLED_FILE.exists():
    acled = pd.read_csv(ACLED_FILE)
    acled.columns = acled.columns.str.lower()
    acled["event_date"] = pd.to_datetime(acled["event_date"])
    acled = acled[acled["event_type"] == "Protests"]
    # regex from session 6: pull the crowd-size bucket out of the tags column
    acled["crowd_size"] = acled["tags"].str.extract(r"crowd size=([^;]+)", expand=False).str.strip()
    print(acled.shape)
    print(acled["crowd_size"].value_counts())
else:
    print("Download your ACLED export first and save it as", ACLED_FILE)

In [ ]:
if ACLED_FILE.exists():
    weekly = (acled.groupby(["event_date", "country"]).size()
                   .unstack("country", fill_value=0)
                   .resample("W").sum())
    big = acled[acled["crowd_size"].isin(["large", "massive"])]
    weekly_big = (big.groupby(["event_date", "country"]).size()
                     .unstack("country", fill_value=0)
                     .resample("W").sum())
    ax = weekly.rolling(4).mean().plot(figsize=(10, 4))
    ax.set_title("Protest events per week (4-week rolling mean)")
    ax.set_ylabel("Events")
    print(weekly_big.tail())

##### Exercise 6

1. Which countries had the most **large or massive** protests per week in your export? Normalise by population (World Bank data) before you compare.
2. ACLED counts **events**, the French counts measure **people**. How would you combine the two in your analysis without mixing them up?
3. Check how many French protest events ACLED records on the 2023 days in section 1. Does ACLED's crowd-size bucket agree with the two counts?

In [ ]:
# Write your exercise here

### 6. From this notebook to your project

- [ ] Pick **one** research question and write your hypothesis in one sentence (discuss it with us if it is your own topic)
- [ ] Collect the missing days and keep a `source` for every number
- [ ] Put your code in a GitHub repository with a detailed README (data sources, how to run, limits)
- [ ] Write the short article-style report and record the 3-minute video
- [ ] Submit on Moodle by **18 December** and add us as collaborators on GitHub